In [17]:
from dotenv import load_dotenv, find_dotenv
from pprint import pprint

In [18]:

load_dotenv(find_dotenv(), override=True)

True

In [19]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [20]:
# give your tool a custom name and description by explicitly specifying it
# if you do not specify a custom name, the tool will use the function name as its name
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

# Entering doc string is important if you are not providing a custom name and description for the tool
# The doc string will be used as the tool's description in that case

In [21]:
# check the type of the tool
print(type(square_root))


<class 'langchain_core.tools.structured.StructuredTool'>


In [22]:
print(square_root.args)

{'x': {'title': 'X', 'type': 'number'}}


Now that we know how arguments are to be passes, we can invoke the tool in the following ways:

In [23]:
# invoke the tool by the given name 
square_root.invoke({"x": 16})

4.0

In [24]:
# or by the implicit name (function nanem)
tool1.invoke({"x": 64})

8.0

In [25]:
import langchain
print(langchain.__version__)

1.5.0


In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [27]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately 21.61018278497431.


In [28]:
pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='ac5df7a0-1c7c-4a7a-98b0-9543112db751'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2008, 'prompt_tokens': 158, 'total_tokens': 2166, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1984, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Em9Z3HEbTHWHbz5iDnjae7kEVcl7d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f6fe-3667-7052-9ce9-a718c5d9f07c-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_jlpMzRjOPdhVcazdD5jAZQjU', 'type': 'tool_call'}

In [29]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_jlpMzRjOPdhVcazdD5jAZQjU', 'type': 'tool_call'}]


<small>

```
.tool_calls 
```

Is a list containing the tool invocations the model requested. It tells which tool the LLM selected, what arguments it passed and the internal tool call ID.

</small>

In [30]:
print(response["messages"][2].content)

21.61018278497431


In [31]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

I’m trained on a mix of data with a knowledge cutoff around September 2023 (some deployments may include a bit of newer info, but I don’t have real-time browsing by default). I don’t automatically access new events or data after that unless you provide details or enable a browsing tool. If you share up-to-date info, I can help reason about it or verify against what you provide. And if you need math, I can compute square roots and squares exactly using my tools.


# Web Search

In [33]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
import os
from pprint import pprint

# Tavily's search client for performing web searches
# You can run this without setting the API key but that has a very small anonymous quota associated with your account
# So it is better to use an API key for higher quota and reliability
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Daniel_Lurie',
   'title': 'Daniel Lurie - Wikipedia',
   'content': 'Daniel Lawrence Lurie (born February 4, 1977) is an American politician and philanthropist who is the 46th and incumbent mayor of San Francisco, serving since 2025. He was elected in 2024, defeating then-incumbent London Breed. He is the founder and former CEO of Tipping Point Community. He has become known for his use of social media as mayor.\n\n## Early life\n\n[edit] [...] | Lurie in 2024 |\n| 46th Mayor of San Francisco |\n| Incumbent |\n| Assumed office  January 8, 2025 |\n| Preceded by | London Breed |\n|  |\n| Personal details |\n| Born | Daniel Lawrence Lurie  (1977-02-04) February 4, 1977 (age 49)  San Francisco, California, U.S. |\n| Party | Democratic "Democratic Party (United States)") |\n| Spouse | Becca Prowda   \u200b (m. 2006)\u200b 

<small>

```python
def web_search(query: str) -> Dict[str, Any]:
```

The function definition means the input will be string and the output will be a dictionary which will have string keys and can contain values of any type (Any).

The function could also simply be written as the following: 

```python
def web_search(query):
    return tavily_client.search(query)
```

The previous way just gives type hints and provides extra information.

</small>

In [34]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [35]:
pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='4a7bf9a8-5a72-445e-af4a-6446c0d97814'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 224, 'prompt_tokens': 133, 'total_tokens': 357, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Em9ZwlvKZ190LhLBEwLAOjbX7Hruq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f6ff-0c4a-7215-a885-1032d374aa1c-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2026 London Breed'}, '

In [36]:
pprint(response['messages'][-1].content)

('Daniel Lurie is the current mayor of San Francisco. He took office in '
 'January 2025 after winning the 2024 election; London Breed is the former '
 'mayor. If you’d like, I can pull up the latest official bio or news '
 'confirmation.')


In [37]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'current mayor of San Francisco 2026 London Breed'},
  'id': 'call_XcqsHDlrL1DTJGM4shsLsN4v',
  'name': 'web_search',
  'type': 'tool_call'}]


To see all tools called with their name, arguments and ID, do the following:

In [38]:
for msg in response["messages"]:
    if hasattr(msg, "tool_calls"):
        for tc in msg.tool_calls:
            print("Tool:", tc["name"])
            print("Args:", tc["args"])
            print("Tool ID:", tc["id"])
            print()

Tool: web_search
Args: {'query': 'current mayor of San Francisco 2026 London Breed'}
Tool ID: call_XcqsHDlrL1DTJGM4shsLsN4v

